# Formica KG Query Pipeline — Entity Resolution, Template Classification, and Cypher Execution

End-to-end demo matching `run_batch_pipeline.py`:

1. **NER** (`dslim/bert-base-NER`, optional `NER_MODEL` env override with fallback)
2. **Entity linking** — gazetteer + aliases + semantic search (two-pass with template-aware types)
3. **Formica template classification** — 21-class SVM on masked query
4. **Template instantiation** — triplet slots → Cypher (propagation, transit, single-entity fallbacks)
5. **Neo4j execution** + optional Gemini summarization

In [55]:
%pip install transformers torch pandas


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Entity Resolver / Entity Linker

Gazetteer + `kg_entity_aliases.json` + BERT NER + semantic similarity (`all-MiniLM-L6-v2`).  
The full pipeline runs **two entity-resolution passes** — the second uses Formica template type preferences (see final cell).

In [96]:
import os
from transformers import pipeline

# Same defaults as run_batch_pipeline.py
DEFAULT_NER_MODEL = "dslim/bert-base-NER"
# gouravsinha/finance-NER is an MPT-7B LLM, not a token-classification NER model — do not use with pipeline("ner").
# Optional: NER_MODEL=musk1209/finsight-ner for finance-tuned BERT NER.


def _load_ner_pipeline(model_name: str):
    return pipeline(
        "ner",
        model=model_name,
        aggregation_strategy="simple",
        device=-1,
    )


ner_model = os.getenv("NER_MODEL", DEFAULT_NER_MODEL)
try:
    ner_pipeline = _load_ner_pipeline(ner_model)
    if ner_model != DEFAULT_NER_MODEL:
        print(f"NER model: {ner_model}")
except (ValueError, OSError, AttributeError) as exc:
    if ner_model == DEFAULT_NER_MODEL:
        raise
    print(
        f"Warning: failed to load NER model '{ner_model}' ({exc}). "
        f"Falling back to {DEFAULT_NER_MODEL}."
    )
    ner_pipeline = _load_ner_pipeline(DEFAULT_NER_MODEL)

#query = "What is the transmission channel from refinery outage to Refining?"
# query = "How much Indian carbon black transits through Cape of Good Hope?"
# query = "How does a shock at Russia propagate to Larsen & Toubro?"
query = "if there is an increase in spending in AI infrastructure,  which all companies will benefit from that"
entities = ner_pipeline(query)
for entity in entities:
    print(
        f"Entity: {entity['word']:<20} | Type: {entity['entity_group']:<10} "
        f"| Confidence: {entity['score']:.4f}"
    )

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


Entity: AI                   | Type: MISC       | Confidence: 0.9869


In [97]:
from neo4j import GraphDatabase
import pandas as pd

driver = GraphDatabase.driver(
    "bolt://localhost:7687",
    auth=("neo4j", "admin1234")
)

def load_nodes(tx):
    q='''
    MATCH (n)
    OPTIONAL MATCH (n)-[r]->(m)
    RETURN n.id as id, n.label as name,
            n.type as node_type,
           type(r) as edge_type,
           collect(type(r)+': '+coalesce(m.label,'')) as relationships
    '''
    return list(tx.run(q))
with driver.session() as s:
    nodes=s.execute_read(load_nodes)

In [98]:
node_docs=[]
for r in nodes:
    node_type = r['node_type']
    node_name = r['name']
    node_docs.append({'id':r['id'], 'node_type': node_type, 'node_name':node_name})

df = pd.DataFrame(node_docs)
df.to_csv("data_node_names.csv", index=False)

print("Saved to data.csv")

Saved to data.csv


In [99]:
%pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

import importlib
import entity_resolver
import formica_template_classes
importlib.reload(formica_template_classes)
importlib.reload(entity_resolver)
from entity_resolver import load_aliases, resolve_entities

embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
aliases = load_aliases()

node_df = df.dropna(subset=["node_name"]).drop_duplicates(subset=["node_name"]).reset_index(drop=True)
node_names = node_df["node_name"].astype(str).tolist()
node_emb = embedder.encode(node_names, convert_to_tensor=True, normalize_embeddings=True)

# First-pass entity linking (template-agnostic). A second pass runs after classification.
entity_texts, matches_df = resolve_entities(
    query, ner_pipeline, embedder, node_df, node_emb, aliases=aliases
)
print("Entities:", entity_texts)
print(matches_df.to_string(index=False))




[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Entities: ['AI']
entity matched_node_name matched_id matched_type  similarity   source
    AI     Generative AI    T_GENAI   TECHNOLOGY     0.64106 semantic


In [18]:
from google import genai

# Initialize Gemini client
import os
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))


def summarize_hops(hop_path: str) -> str:
    prompt = f"""
You are a financial knowledge-graph analyst.

Summarize the following multi-hop relationship path from a knowledge graph.

Requirements:
1. Identify the starting entity and ending entity.
2. Explain the path in simple business/financial language.
5. Mention important numerical facts if present.
6. Do not invent relationships or facts.
8. Keep the summary to 2-4 sentences.

Knowledge graph path:

{hop_path}
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return response.text


if not hop_path:
    raise ValueError("No hop path was saved. Re-run the shortest-path cell first.")

summary = summarize_hops(hop_path)

print(summary)

The knowledge graph path connects the geographic bottleneck of the **Strait of Hormuz** to the financial institution **ICICI Lombard**. Geopolitically, 41-53% of India's crude oil imports transit the Strait of Hormuz to supply producers like Oil & Natural Gas Corp, whose realizations track Brent crude prices. In turn, rising Brent crude prices negatively impact Cholamandalam Investment, which belongs to the Banks & NBFCs sector alongside ICICI Lombard.


In [100]:
import re
import numpy as np

def mask_entities_with_types(text: str, entity_types: list[tuple[str, str]]) -> str:
    """Replace matched entity spans with typed tags, e.g. <COMPANY>, <GEOGRAPHY>."""
    masked = text
    for entity, entity_type in sorted(entity_types, key=lambda x: len(x[0]), reverse=True):
        if not entity or not entity_type:
            continue
        tag = f"<{entity_type.upper()}>"
        pattern = re.compile(re.escape(entity), flags=re.IGNORECASE)
        masked = pattern.sub(tag, masked)
    return masked


entity_types = [
    (row["entity"], row["matched_type"])
    for _, row in matches_df.dropna(subset=["entity", "matched_type"]).iterrows()
]
masked_query = mask_entities_with_types(query, entity_types)

print("Original query:", query)
print("Entity masks:  ", entity_types)
print("Masked query:  ", masked_query)

query_embedding = embedder.encode(masked_query, normalize_embeddings=True)
print(f"\nEmbedding shape: {query_embedding.shape}")
print(f"Embedding (first 8 dims): {np.round(query_embedding[:8], 4)}")

Original query: if there is an increase in spending in AI infrastructure,  which all companies will benefit from that
Entity masks:   [('AI', 'TECHNOLOGY')]
Masked query:   if there is an increase in spending in <TECHNOLOGY> infrastructure,  which all companies will benefit from that

Embedding shape: (384,)
Embedding (first 8 dims): [ 0.0033 -0.0359  0.0317 -0.0669  0.0647 -0.017   0.0051 -0.0083]


In [93]:
# Updated cell 11 to load only edges that appear on paths between the matched entities (from matches_df), instead of the full graph.

# What changed:

# Uses matched node IDs from NER resolution (e.g. G_HORMUZ ↔ C_ICICIGI)
# Finds all paths up to MAX_PATH_HOPS (12) between each matched pair
# Collects distinct edges on those paths
# Saves the filtered set to data_edge_names.csv
# For your ICICI Lombard / Hormuz example, this loads:

# 320 edges across 16 edge types (e.g. TRANSITS, HURT_BY, CAUSES, CONTAINS)
MAX_PATH_HOPS = 12

matched_nodes = (
    matches_df.dropna(subset=["matched_id", "matched_node_name"])
    .drop_duplicates(subset=["matched_id"])
    .reset_index(drop=True)
)
if len(matched_nodes) < 2:
    raise ValueError("Need at least two matched nodes to load connecting edges.")

entity_pairs = [
    (matched_nodes.iloc[i]["matched_id"], matched_nodes.iloc[j]["matched_id"])
    for i in range(len(matched_nodes))
    for j in range(i + 1, len(matched_nodes))
]


def load_edges_between(tx, source_id, target_id, max_hops):
    q = f"""
    MATCH (source:Node {{id: $source_id}}), (target:Node {{id: $target_id}})
    MATCH p = (source)-[*1..{max_hops}]-(target)
    UNWIND relationships(p) AS r
    WITH DISTINCT r, startNode(r) AS a, endNode(r) AS b
    RETURN
        a.id AS source,
        b.id AS target,
        type(r) AS relation,
        a.label AS source_name,
        b.label AS target_name,
        a.type AS source_type,
        b.type AS target_type
    """
    return list(tx.run(q, source_id=source_id, target_id=target_id))


edges = []
with driver.session() as s:
    for src_id, tgt_id in entity_pairs:
        src_name = matched_nodes.loc[matched_nodes["matched_id"] == src_id, "matched_node_name"].iloc[0]
        tgt_name = matched_nodes.loc[matched_nodes["matched_id"] == tgt_id, "matched_node_name"].iloc[0]
        print(f"Loading edges on paths between {src_name} ({src_id}) and {tgt_name} ({tgt_id})")
        edges.extend(s.execute_read(load_edges_between, src_id, tgt_id, MAX_PATH_HOPS))

seen = set()
edge_docs = []
for e in edges:
    key = (e["source"], e["target"], e["relation"])
    if key in seen:
        continue
    seen.add(key)
    edge_docs.append({
        "source": e["source"],
        "target": e["target"],
        "source_name": e["source_name"],
        "target_name": e["target_name"],
        "source_type": e["source_type"],
        "target_type": e["target_type"],
        "edge_type": e["relation"],
    })

df1 = pd.DataFrame(edge_docs)
df1.to_csv("data_edge_names.csv", index=False)

print(f"\nLoaded {len(df1)} distinct edges across {df1['edge_type'].nunique()} edge types")
print("Edge types:", sorted(df1["edge_type"].unique().tolist()))



Loading edges on paths between Strait of Hormuz disruption (E_HORMUZ) and Gross refining margin (M_GRM)

Loaded 343 distinct edges across 17 edge types
Edge types: ['BENEFITS_FROM', 'CAUSES', 'CONSTRAINED_BY', 'CONSTRAINS', 'CONTAINS', 'CUSTOMER_OF', 'ENABLES', 'EXPOSED_TO_ORDERBOOK', 'HEDGES', 'HURT_BY', 'NEEDS', 'OWNS', 'PRICE_LINKED_TO', 'PRODUCES', 'SOURCED_FROM', 'SUPPLIES_TO', 'TRANSITS']


In [101]:
# Embed unique edge types and compare with the masked query embedding
edge_df = (
    df1.dropna(subset=["edge_type"])
    .drop_duplicates(subset=["edge_type"])
    .reset_index(drop=True)
)
edge_types = edge_df["edge_type"].astype(str).tolist()

# Make relation names more readable for the embedder (e.g. HURT_BY -> hurt by)
edge_labels = [t.replace("_", " ").lower() for t in edge_types]

query_emb = embedder.encode(masked_query, convert_to_tensor=True, normalize_embeddings=True)
edge_emb = embedder.encode(edge_labels, convert_to_tensor=True, normalize_embeddings=True)

top_k = min(1, len(edge_types))
hits = util.semantic_search(query_emb, edge_emb, top_k=top_k)[0]

edge_matches = []
for hit in hits:
    row = edge_df.iloc[hit["corpus_id"]]
    edge_matches.append({
        "edge_type": row["edge_type"],
        "edge_label": edge_labels[hit["corpus_id"]],
        "similarity": float(hit["score"]),
    })

edge_matches_df = pd.DataFrame(edge_matches)
print("Top edge-type matches for masked query:")
print(edge_matches_df.to_string(index=False))

Top edge-type matches for masked query:
    edge_type    edge_label  similarity
BENEFITS_FROM benefits from    0.383352


In [7]:
# Stock impact query-template dataset for DeBERTa fine-tuning
import json
from pathlib import Path

DATA_DIR = Path("data")
TRAIN_CSV = DATA_DIR / "deberta_stock_impact_train.csv"
TEST_CSV = DATA_DIR / "deberta_stock_impact_test.csv"
LABELS_JSON = DATA_DIR / "deberta_stock_impact_labels.json"

# Regenerate larger synthetic set (100 examples/class -> ~1200 train / ~300 test)
%run generate_query_template_dataset.py --examples-per-template 100

with open(LABELS_JSON) as f:
    label_meta = json.load(f)

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
label2id = label_meta["label2id"]
id2label = {int(k): v for k, v in label_meta["id2label"].items()}

print(f"Train: {len(train_df)} | Test: {len(test_df)} | Labels: {label_meta['num_labels']}")
print(train_df["label"].value_counts().sort_index())

Wrote 1200 train / 300 test examples
Per-class train counts:
label
T_Beneficiary           80
T_CausalChain           80
T_CompareExposure       80
T_ConstraintRisk        80
T_EventScenario         80
T_Hedging               80
T_InvestmentFlow        80
T_MacroTransmission     80
T_OwnershipStructure    80
T_PriceLinkage          80
T_RegulatoryImpact      80
T_SectorExposure        80
T_StockImpact           80
T_SupplyDisruption      80
T_TransitRisk           80

Per-class test counts:
label
T_Beneficiary           20
T_CausalChain           20
T_CompareExposure       20
T_ConstraintRisk        20
T_EventScenario         20
T_Hedging               20
T_InvestmentFlow        20
T_MacroTransmission     20
T_OwnershipStructure    20
T_PriceLinkage          20
T_RegulatoryImpact      20
T_SectorExposure        20
T_StockImpact           20
T_SupplyDisruption      20
T_TransitRisk           20
Train: 1200 | Test: 300 | Labels: 15
label
T_Beneficiary           80
T_CausalChain          

In [8]:
%pip install -q 'accelerate>=0.26.0' datasets evaluate scikit-learn

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import numpy as np
import evaluate

MODEL_NAME = "microsoft/deberta-v3-small"
OUTPUT_DIR = "models/deberta-stock-impact-template"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

def encode_labels(example):
    example["labels"] = label2id[example["labels"]]
    return example

train_ds = Dataset.from_pandas(train_df).map(tokenize_batch, batched=True)
test_ds = Dataset.from_pandas(test_df).map(tokenize_batch, batched=True)

train_ds = train_ds.rename_column("label", "labels").map(encode_labels)
test_ds = test_ds.rename_column("label", "labels").map(encode_labels)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        learning_rate=3e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        logging_steps=20,
        report_to="none",
    ),
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
metrics = trainer.evaluate()
print(metrics)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Map: 100%|██████████| 300/300 [00:00<00:00, 53888.27 examples/s]
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Library/Frameworks/Python.framework/Versions/

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,2.692000,2.307244,0.330000,0.246802
2,1.251500,0.750582,0.910000,0.906867
3,0.418400,0.227972,0.970000,0.969578
4,0.141100,0.093439,0.976667,0.976474
5,0.090500,0.062838,0.986667,0.986662


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pin

{'eval_loss': 0.06283844262361526, 'eval_accuracy': 0.9866666666666667, 'eval_f1_macro': 0.9866624973942048, 'eval_runtime': 0.6118, 'eval_samples_per_second': 490.35, 'eval_steps_per_second': 31.055, 'epoch': 5.0}


('models/deberta-stock-impact-template/tokenizer_config.json',
 'models/deberta-stock-impact-template/special_tokens_map.json',
 'models/deberta-stock-impact-template/spm.model',
 'models/deberta-stock-impact-template/added_tokens.json',
 'models/deberta-stock-impact-template/tokenizer.json')

In [82]:
from formica_template_classifier import FormicaTemplateClassifier

formica_classifier = FormicaTemplateClassifier()
formica_classifier.load()

def classify_stock_query(text: str):
    """Classify into one of 21 Formica template classes (PoS + keywords + SVM)."""
    return formica_classifier.predict_proba_like(text, top_k=3)

# Sample queries covering Formica logical/comparative/quantitative/simple classes
sample_queries = [
    # T_StockImpact
    "Why are ICICI Lombard stocks impacted by Hormuz disruption?",
    "How exposed is Reliance Industries to crude price rally?",
    # T_CausalChain
    "Trace the path from Strait of Hormuz to ICICI Lombard via Brent crude price.",
    "How does Hormuz disruption lead to higher India CPI?",
    # T_Beneficiary
    "Which companies benefit from higher Brent crude price?",
    "Who wins from marine war-risk premium spike in Indian markets?",
    # T_CompareExposure
    "Compare ONGC and Oil India exposure to oil supply shock.",
    "Which is more hurt by Brent rally: Tata Motors or Maruti Suzuki?",
    # T_ConstraintRisk
    "What constrains NTPC growth in AI infrastructure capex?",
    "Which bottlenecks limit data centre expansion in India?",
    # T_EventScenario
    "What if Strait of Hormuz closes completely?",
    "Stress test ICICI Lombard for prolonged Hormuz disruption.",
    # T_Hedging
    "How does Reliance Industries hedge against USD/INR volatility?",
    "What natural hedge does Federal Bank have for Gulf job risk?",
    # T_InvestmentFlow
    "Who invests in data centre ecosystem in India?",
    "Which PE firms are backing Indian AI infrastructure plays?",
    # T_MacroTransmission
    "How does RBI policy rate affect Bajaj Finance?",
    "What happens to Indian Oil when Brent crude price rises?",
    # T_OwnershipStructure
    "Who owns Hindustan Petroleum?",
    "What subsidiaries does Oil & Natural Gas Corp have?",
    # T_PriceLinkage
    "How is Asian Paints linked to Brent crude price?",
    "Does Petronet LNG track spot LNG price?",
    # T_RegulatoryImpact
    "How is Indian Oil affected by LPG subsidy policy?",
    "What regulatory risk does Chambal Fertilisers face?",
    # T_SectorExposure
    "Which companies are in Banks & NBFCs sector?",
    "List Oil & Gas constituents exposed to Hormuz disruption.",
    # T_SupplyDisruption
    "How does Hormuz disruption disrupt crude oil supply?",
    "What feedstock shortage risk does Asian Paints face from naphtha?",
    # T_TransitRisk
    "How much Indian crude oil transits through Strait of Hormuz?",
    "What share of India's LNG imports passes through Hormuz?",
]

for q in sample_queries:
    print(f"\nQuery: {q}")
    for label, score in classify_stock_query(q):
        print(f"  {label:<22} {score:.4f}")


Query: Why are ICICI Lombard stocks impacted by Hormuz disruption?
  F_Simple               0.5929
  F_CompMore             0.0863
  F_QuantMin             0.0715

Query: How exposed is Reliance Industries to crude price rally?
  F_Simple               0.6341
  F_LogUnion             0.0714
  F_QuantCount           0.0709

Query: Trace the path from Strait of Hormuz to ICICI Lombard via Brent crude price.
  F_Simple               0.6844
  F_QuantCount           0.0664
  F_CompMore             0.0636

Query: How does Hormuz disruption lead to higher India CPI?
  F_CompMore             0.5273
  F_Simple               0.1427
  F_QuantMin             0.0757

Query: Which companies benefit from higher Brent crude price?
  F_CompMore             0.4694
  F_Simple               0.1420
  F_QuantCount           0.0846

Query: Who wins from marine war-risk premium spike in Indian markets?
  F_Simple               0.6242
  F_CompMore             0.1171
  F_LogIntersection      0.0703

Query: Com

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator CountVectorizer from version 1.7.2 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying to unpickle estimator FeatureUnion from version 1.7.2 when using version 1.7.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/base.py:440: InconsistentVersionWarning: Trying

In [105]:
import importlib
import json
import re

import entity_resolver
import formica_template_classes
import formica_template_resolver

importlib.reload(formica_template_classes)
importlib.reload(entity_resolver)
importlib.reload(formica_template_resolver)

from entity_resolver import load_aliases, resolve_entities
from formica_template_resolver import (
    enrich_for_formica_template,
    execute_formica_template,
    expand_formica_template,
)


def mask_entities_with_types(text: str, entity_types: list[tuple[str, str]]) -> str:
    masked = text
    for entity, entity_type in sorted(entity_types, key=lambda x: len(x[0]), reverse=True):
        if not entity or not entity_type:
            continue
        tag = f"<{entity_type.upper()}>"
        pattern = re.compile(re.escape(entity), flags=re.IGNORECASE)
        masked = pattern.sub(tag, masked)
    return masked

def format_kg_rows_for_summary(rows, expanded):
    """Turn Neo4j result rows into hop-style text for Gemini summarization."""
    if not rows:
        return ""

    lines = [
        f"Template: {expanded['template_id']} ({expanded['template_label']})",
        f"Query intent: {expanded['description']}",
        "",
    ]

    for i, row in enumerate(rows, start=1):
        data = dict(row)
        if "path_names" in data and "rel_types" in data:
            names = data.get("path_names") or []
            rels = data.get("rel_types") or []
            header = (
                f"Path {i}: {data.get('source_name', names[0] if names else '?')}"
                f" -> {data.get('target_name', names[-1] if names else '?')}"
                f" ({data.get('hops', len(rels))} hops)"
            )
            lines.append(header)
            for j, rel in enumerate(rels):
                left = names[j] if j < len(names) else "?"
                right = names[j + 1] if j + 1 < len(names) else "?"
                lines.append(f"  {left} -[{rel}]-> {right}")
        elif "relationship" in data and "parent_name" in data:
            lines.append(
                f"Row {i}: {data.get('parent_name')} -[{data['relationship']}]-> "
                f"{data.get('child_name')}"
            )
        elif "commodity_name" in data and ("geography_name" in data or "chokepoint" in data):
            geo = data.get("geography_name") or data.get("chokepoint")
            share = f" share={data['share']}" if data.get("share") is not None else ""
            extra = f" [{data['channel']}]" if data.get("channel") else ""
            lines.append(
                f"Row {i}: {data['commodity_name']} -[TRANSITS]-> {geo}{share}{extra}"
            )
        else:
            parts = [f"{k}={v}" for k, v in data.items() if v is not None]
            lines.append(f"Row {i}: " + " | ".join(parts))

    return "\n".join(lines)


# Pipeline (same order as run_batch_pipeline.py):
# 1. resolve entities (first pass — from cell above)
# 2. mask -> classify
# 3. resolve entities again with template_label (second pass)
# 4. re-mask -> enrich -> expand Cypher -> execute -> summarize
# query = "How much Indian carbon black transits through Cape of Good Hope?"
# query = "How does a shock at Russia propagate to Larsen & Toubro?"

aliases = load_aliases()

if matches_df.empty:
    raise ValueError("No entities resolved — run the entity resolution cell first.")

entity_types = [
    (row["entity"], row["matched_type"]) for _, row in matches_df.iterrows()
]
masked_query = mask_entities_with_types(query, entity_types)
top_template, confidence = formica_classifier.predict(masked_query)

# Second-pass linking with template-aware type preferences
_, matches_df = resolve_entities(
    query,
    ner_pipeline,
    embedder,
    node_df,
    node_emb,
    aliases=aliases,
    template_label=top_template,
)
if matches_df.empty:
    raise ValueError("No entities resolved after template-aware second pass.")

entity_types = [
    (row["entity"], row["matched_type"]) for _, row in matches_df.iterrows()
]
masked_query = mask_entities_with_types(query, entity_types)

print(f"Query: {query}")
print(f"Masked: {masked_query}")
print(f"Formica template: {top_template} ({confidence:.4f})")
print(f"Resolved entities ({len(matches_df)}):")
print(matches_df[["entity", "matched_node_name", "matched_id", "matched_type"]].to_string(index=False))
print()

matches_df = enrich_for_formica_template(
    query, top_template, matches_df, node_df, aliases=aliases
)
expanded = expand_formica_template(query, top_template, matches_df)
print(json.dumps(expanded, indent=2))

kg_rows, strategy = execute_formica_template(driver, expanded)
print(f"\nRows returned: {len(kg_rows)} (strategy: {strategy})")

if not kg_rows and expanded.get("missing_parameters"):
    print("Missing parameters:", expanded["missing_parameters"])
elif kg_rows:
    for row in kg_rows[:5]:
        print(dict(row))

    kg_hop_path = format_kg_rows_for_summary(kg_rows, expanded)
    if kg_hop_path:
        print("\n--- KG rows formatted for summarizer ---\n")
        print(kg_hop_path)
        template_summary = summarize_hops(kg_hop_path)
        print("\n--- Gemini summary ---\n")
        print(template_summary)
    else:
        print("\nNo rows to summarize.")
else:
    print("\nNo KG rows returned.")

Query: if there is an increase in spending in AI infrastructure,  which all companies will benefit from that
Masked: if there is an increase in spending in <PRODUCT> infrastructure,  which all companies will benefit from that
Formica template: F_Simple (0.8064)
Resolved entities (1):
entity      matched_node_name matched_id matched_type
    AI AI server / GPU system P_AISERVER      PRODUCT

{
  "template_label": "F_Simple",
  "template_id": "simple_triplet",
  "template_class": "F_Simple",
  "description": "Simple question: subject linked to objects via a predicate.",
  "cypher": "MATCH (company:Node {type: 'COMPANY'})-[r:BENEFITS_FROM]->(shock:Node {id: $nnp1})\nRETURN company.label AS company_name, shock.label AS shock_name, type(r) AS relationship\nORDER BY company_name",
  "parameters": {
    "nnp1": "P_AISERVER"
  },
  "triplet_slots": {
    "nnp1": "P_AISERVER",
    "nnp2": null,
    "nnp3": null,
    "prop1": "BENEFITS_FROM",
    "nn1": "PRODUCT",
    "nn2": null,
    "threshold